# Part 1: Tensor fundamentals

Deep-learning code is largely code for transforming batches of tensors. In this notebook we will concentrate on the information carried by a tensor - its **shape**, **dtype**, and **device**. We'll also look at making transformations that preserve the intended meaning of its axes.

For every task, try to predict the shape and dtype of the result before running the cell.

## Setup

Colab normally provides PyTorch already. The following cell also selects an accelerator when one is available. Nothing in this notebook requires a GPU.

In [1]:
import torch

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"PyTorch version: {torch.__version__}")
print(f"Selected device: {device}")

PyTorch version: 2.11.0+cpu
Selected device: cpu


## Shape, dtype, and device

A tensor does not know that an axis means ‘examples’, ‘features’, or ‘classes’. That meaning comes from the program we write. We will use the convention that the first axis is the batch axis unless stated otherwise.

The following tensor contains four examples, each represented by three floating-point features.

In [2]:
features = torch.tensor(
    [[1.2, -0.3, 2.1],
     [0.7,  1.4, 0.2],
     [2.3,  0.1, 1.7],
     [0.2, -1.1, 0.8]],
    dtype=torch.float32,
)
labels = torch.tensor([2, 0, 1, 2], dtype=torch.int64)

print(features)
print("features:", features.shape, features.dtype, features.device)
print("labels:  ", labels.shape, labels.dtype, labels.device)

tensor([[ 1.2000, -0.3000,  2.1000],
        [ 0.7000,  1.4000,  0.2000],
        [ 2.3000,  0.1000,  1.7000],
        [ 0.2000, -1.1000,  0.8000]])
features: torch.Size([4, 3]) torch.float32 cpu
labels:   torch.Size([4]) torch.int64 cpu


### Task: create tensors deliberately

Create:

- a `5 × 4` floating-point tensor called `x`, drawn from a standard Normal distribution;
- a length-five integer tensor called `targets`, containing the values `0, 1, 1, 0, 1`; and
- a Boolean tensor called `positive` indicating which entries of `x` are positive.

Use factory functions such as `torch.randn` and `torch.tensor`, and specify dtypes when their choice matters.

In [3]:
# TODO: replace the three None values.
x = torch.randn(5, 4)
targets = torch.tensor([0, 1, 1, 0, 1])
positive = x > 0


In [4]:
# Run these checks after completing Task 1.
assert isinstance(x, torch.Tensor) and x.shape == (5, 4)
assert x.dtype == torch.float32
assert targets.shape == (5,) and targets.dtype == torch.int64
assert positive.shape == x.shape and positive.dtype == torch.bool
print("Task 1 checks passed")

Task 1 checks passed


## Indexing and reductions

Indexing can remove an axis; slicing normally preserves it. Compare `features[0]` with `features[0:1]` before running the following cell.

In [5]:
first_example = features[0]
first_example_as_batch = features[0:1]
second_feature = features[:, 1]

print(first_example.shape)
print(first_example_as_batch.shape)
print(second_feature.shape)

torch.Size([3])
torch.Size([1, 3])
torch.Size([4])


Reductions such as `sum` and `mean` remove the reduced axis by default. `keepdim=True` retains it with size one, which is often useful for later broadcasting.

In [6]:
feature_means = features.mean(dim=0)
example_means = features.mean(dim=1)
example_means_column = features.mean(dim=1, keepdim=True)
overall_mean = features.mean()

print("feature means:        ", feature_means.shape)
print("example means:        ", example_means.shape)
print("example means column: ", example_means_column.shape)
print("overall mean:         ", overall_mean.shape, overall_mean.item())

feature means:         torch.Size([3])
example means:         torch.Size([4])
example means column:  torch.Size([4, 1])
overall mean:          torch.Size([]) 0.7750000357627869


### Task: centre a batch of features

Compute the mean of each feature over the batch and subtract it from every example. Store the result as `centred`. Do not write a loop.

Afterwards, explain why a tensor of shape `(3,)` can be subtracted from one of shape `(4, 3)`.

In [7]:
centred = features - features.mean(dim=0)


In [8]:
assert centred.shape == features.shape
torch.testing.assert_close(centred.mean(dim=0), torch.zeros(3), atol=1e-6, rtol=0)
print("Task 2 checks passed")

Task 2 checks passed


## Reshaping and permuting axes

Suppose `images` represents a batch of 12 RGB images, each 28 pixels high and 28 pixels wide. PyTorch image models normally use `(batch, channels, height, width)`.

In [9]:
images = torch.randn(12, 3, 28, 28)
flat_images = images.reshape(12, -1)
channels_last = images.permute(0, 2, 3, 1)

print("images:        ", images.shape)
print("flat images:   ", flat_images.shape)
print("channels last: ", channels_last.shape)

images:         torch.Size([12, 3, 28, 28])
flat images:    torch.Size([12, 2352])
channels last:  torch.Size([12, 28, 28, 3])


`reshape` changes how elements are grouped into axes. `permute` changes the order of existing axes. Neither operation changes the values themselves.

### Task: restore the image batch

Starting only from `channels_last`, recover a tensor called `restored` with shape `(12, 3, 28, 28)` and verify that it equals `images`.

In [10]:
restored = channels_last.permute(0, 3, 1, 2)
torch.testing.assert_close(restored, images)

In [11]:
assert restored.shape == images.shape
torch.testing.assert_close(restored, images)
print("Task 3 checks passed")

Task 3 checks passed


## Devices

Operations require their tensor operands to be on compatible devices. Prefer device-agnostic code over calls such as `.cuda()`, because the same notebook can then run on a CPU, a CUDA GPU, or another supported accelerator.

In [12]:
x_cpu = torch.randn(1024, 256)
weights_cpu = torch.randn(256, 64)

x_device = x_cpu.to(device)
weights_device = weights_cpu.to(device)
output = x_device @ weights_device

print(output.shape, output.device)

torch.Size([1024, 64]) cpu


### Task: repair the device pattern

The following *pattern* is wrong when a GPU is selected because it moves the data but not the weights:

```python
data = torch.randn(16, 8).to(device)
weights = torch.randn(8, 4)
result = data @ weights
```

Write a corrected version below. It must also work when `device` is the CPU.

In [13]:
data = torch.randn(16, 8).to(device)
weights = torch.randn(8, 4).to(device)
result = data @ weights


## Wrap-up challenge

Implement `standardise_batch(x)` so that each feature has approximately zero mean and unit standard deviation over the batch. Your implementation should:

- work for any two-dimensional floating-point tensor;
- contain no Python loop;
- retain dimensions where that makes the broadcasting intention clearer; and
- add a small `eps` to avoid division by zero.

In [14]:
def standardise_batch(x, eps=1e-6):
    mean = x.mean(dim=0, keepdim=True)
    std = x.std(dim=0, keepdim=True)
    return (x - mean) / (std + eps)


## Check your understanding

1. What information is lost when `(1, 20)` is indexed with `[0]` to produce `(20,)`?
2. Why are class labels normally stored with an integer dtype rather than a floating-point dtype?
3. When is `keepdim=True` useful?
4. Why should device selection occur in one place near the start of a notebook?
5. What does an empty shape, `torch.Size([])`, represent?

## Check your understanding

1. What information is lost when `(1, 20)` is indexed with `[0]` to produce `(20,)`?
2. Why are class labels normally stored with an integer dtype rather than a floating-point dtype?
3. When is `keepdim=True` useful?
4. Why should device selection occur in one place near the start of a notebook?
5. What does an empty shape, `torch.Size([])`, represent?

1. The batch axis. (1, 20) is one example of 20 features; [0] gives (20,) — same numbers, no batch. So later broadcasting lines up from the wrong end. [0:1] keeps it.

2. Labels are identifiers, not quantities. Averaging class 0 and 1 means nothing, and indexing plus losses like CrossEntropyLoss want int64. Floats imply maths you shouldn't be doing, and can silently round to the wrong class.

3. When a reduced result has to line back up with its source. mean(dim=0, keepdim=True) on (N, 3) gives (1, 3) instead of (3,), so it broadcasts safely from the right — and stays correct when you add another axis later.

4. So the decision gets made once. A device mismatch is an error, not a quiet wrong answer; choosing device at the top means everything agrees, and moving between CPU and GPU costs one line.

5. A scalar — zero axes. x.sum() or torch.tensor(5.0) gives torch.Size([]), not (1,). Can't be indexed, broadcasts against anything, still has a dtype and device.